# 面试问题：Process Reward Verifier 怎样识别最终答案正确但中间推理无效的候选，并引导搜索？

        ## 可直接复述的回答主线

        1. Outcome Reward 只看最终答案，无法区分可靠推理和中间错误后猜中答案的轨迹。
2. Process Verifier 对每一步检查操作类型、输入依赖、计算结果和顺序，再聚合为路径分数。
3. 同一案例应让 outcome-only 和 process-guided 在相同候选上选择，并同时报告最终正确率与过程有效率。
4. 过程奖励不能简单按有效步骤求和，否则重复一个正确步骤就能刷高分。
5. 修正需要规定必要 stage、顺序、去重和状态连续性，并保存逐步 verifier 账本。
6. 生产系统还要处理 verifier 偏差、开放式推理等价、对抗性 reward hacking 和成本预算。

        后续实验会用同一批输入依次验证朴素方案、核心机制、失败边界和修正效果。

## 1. 真实案例与输入预览

案例是五张脱敏订单账单，每张包含数量、单价、折扣和应付金额。每题有一条“中间小计错误但最终猜对”的 hack 轨迹和一条步骤连续的 good 轨迹，使 outcome-only 最终正确率看似完美但过程不可审计。

In [1]:
cases = [{"id": "bill-01", "quantity": 3, "unit_price": 120.0, "discount": 0.10}, {"id": "bill-02", "quantity": 5, "unit_price": 48.0, "discount": 0.05}, {"id": "bill-03", "quantity": 2, "unit_price": 399.0, "discount": 0.15}, {"id": "bill-04", "quantity": 8, "unit_price": 25.0, "discount": 0.00}, {"id": "bill-05", "quantity": 4, "unit_price": 88.0, "discount": 0.20}]  # 定义五张具有数量、单价和折扣的脱敏账单。
for case in cases:  # 为每张账单计算评测真值。
    case["truth"] = round(case["quantity"] * case["unit_price"] * (1.0 - case["discount"]), 2)  # 按业务公式得到应付金额。
candidates = {}  # 保存每张账单的 hack 与 good 两条推理轨迹。
for case in cases:  # 为同一输入构造最终答案相同但过程不同的候选。
    subtotal = round(case["quantity"] * case["unit_price"], 2)  # 计算合法轨迹的小计。
    wrong_subtotal = round(subtotal + 10.0, 2)  # 故意制造偏高十元的中间小计。
    hack = {"id": "hack", "steps": [{"stage": "subtotal", "claimed": wrong_subtotal}, {"stage": "total", "claimed": case["truth"]}], "final": case["truth"]}  # 构造中间错误后直接猜中最终答案的轨迹。
    good = {"id": "good", "steps": [{"stage": "subtotal", "claimed": subtotal}, {"stage": "total", "claimed": case["truth"]}], "final": case["truth"]}  # 构造状态连续的正确轨迹。
    candidates[case["id"]] = [hack, good]  # 按 hack 在前保存，复现 outcome 平票时的错误选择。
print("教学实验输入：五张订单账单")  # 标记下表是脱敏离线案例。
print("账单      数量  单价  折扣  应付真值  候选")  # 输出账单字段表头。
for case in cases:  # 逐条展示输入与两类候选。
    print(f"{case['id']:<9} {case['quantity']:>4} {case['unit_price']:>5.0f} {case['discount']:>5.0%} {case['truth']:>9.2f}  {[candidate['id'] for candidate in candidates[case['id']]]}")  # 输出当前账单可读字段。

教学实验输入：五张订单账单
账单      数量  单价  折扣  应付真值  候选
bill-01      3   120   10%    324.00  ['hack', 'good']
bill-02      5    48    5%    228.00  ['hack', 'good']
bill-03      2   399   15%    678.30  ['hack', 'good']
bill-04      8    25    0%    200.00  ['hack', 'good']
bill-05      4    88   20%    281.60  ['hack', 'good']


## 2. Baseline / 基线：Outcome-only 只比较最终金额

两条轨迹最终金额都正确，outcome reward 都是一。稳定 tie-break 选择列表第一条 hack，于是最终准确率 100%，过程有效率却为零。

In [2]:
def outcome_reward(case, candidate):  # 仅按最终金额判断候选是否正确。
    return 1.0 if abs(candidate["final"] - case["truth"]) < 1.0e-9 else 0.0  # 返回二元最终奖励。
baseline_choices = []  # 保存 outcome-only 对五张账单的选择。
for case in cases:  # 逐账单比较两个最终答案相同的候选。
    options = candidates[case["id"]]  # 读取当前账单的 hack 和 good 轨迹。
    chosen = max(options, key=lambda candidate: outcome_reward(case, candidate))  # 平票时稳定选择排在前面的 hack。
    baseline_choices.append({"case": case["id"], "chosen": chosen["id"], "outcome": outcome_reward(case, chosen)})  # 保存选择与最终奖励。
print("Baseline Outcome-only 选择")  # 标记当前输出只检查最终金额。
print("账单      chosen  outcome  中间步骤")  # 输出选择结果表头。
for choice in baseline_choices:  # 逐账单展示 outcome 平票后的轨迹。
    selected_candidate = next(candidate for candidate in candidates[choice["case"]] if candidate["id"] == choice["chosen"])  # 找到当前被选轨迹。
    print(f"{choice['case']:<9} {choice['chosen']:<7} {choice['outcome']:>7.1f}  {selected_candidate['steps']}")  # 输出最终正确但中间错误的证据。

Baseline Outcome-only 选择
账单      chosen  outcome  中间步骤
bill-01   hack        1.0  [{'stage': 'subtotal', 'claimed': 370.0}, {'stage': 'total', 'claimed': 324.0}]
bill-02   hack        1.0  [{'stage': 'subtotal', 'claimed': 250.0}, {'stage': 'total', 'claimed': 228.0}]
bill-03   hack        1.0  [{'stage': 'subtotal', 'claimed': 808.0}, {'stage': 'total', 'claimed': 678.3}]
bill-04   hack        1.0  [{'stage': 'subtotal', 'claimed': 210.0}, {'stage': 'total', 'claimed': 200.0}]
bill-05   hack        1.0  [{'stage': 'subtotal', 'claimed': 362.0}, {'stage': 'total', 'claimed': 281.6}]


## 3. 底层实现：检查 stage、状态依赖和计算连续性

Verifier 先检查 subtotal 是否等于数量乘单价，再要求 total 基于候选自己声称的 subtotal 乘折扣。第二步不能跳回真值绕过错误中间状态。

In [3]:
def verify_candidate(case, candidate):  # 对账单推理轨迹执行逐步状态连续性检查。
    ledger = []  # 保存每个 stage 的期望值、声称值和通过状态。
    expected_subtotal = round(case["quantity"] * case["unit_price"], 2)  # 按输入计算小计真值。
    subtotal_step = candidate["steps"][0]  # 读取候选第一步小计。
    subtotal_valid = subtotal_step["stage"] == "subtotal" and abs(subtotal_step["claimed"] - expected_subtotal) < 1.0e-9  # 检查 stage 和小计计算。
    ledger.append({"stage": "subtotal", "expected": expected_subtotal, "claimed": subtotal_step["claimed"], "valid": subtotal_valid})  # 保存第一步 verifier 结果。
    expected_total_from_claim = round(subtotal_step["claimed"] * (1.0 - case["discount"]), 2)  # 从候选自己的中间状态推导下一步期望。
    total_step = candidate["steps"][1]  # 读取候选第二步总额。
    total_valid = total_step["stage"] == "total" and abs(total_step["claimed"] - expected_total_from_claim) < 1.0e-9  # 检查最终计算是否承接上一步。
    ledger.append({"stage": "total", "expected": expected_total_from_claim, "claimed": total_step["claimed"], "valid": total_valid})  # 保存第二步 verifier 结果。
    process_score = sum(item["valid"] for item in ledger) / len(ledger)  # 计算两步过程通过比例。
    outcome = outcome_reward(case, candidate)  # 单独保留最终答案奖励用于双维评测。
    return process_score, outcome, ledger  # 返回过程分、最终分和逐步账本。
first_case = cases[0]  # 选择第一张账单展示 verifier 中间过程。
print("bill-01 Process Verifier 账本")  # 标记下表比较同一输入的两条轨迹。
print("candidate stage      expected  claimed  valid")  # 输出 verifier 账本表头。
for candidate in candidates[first_case["id"]]:  # 逐候选检查 hack 和 good。
    process_score, outcome, ledger = verify_candidate(first_case, candidate)  # 获取当前候选的过程结果。
    for item in ledger:  # 逐步展示状态连续性检查。
        print(f"{candidate['id']:<9} {item['stage']:<9} {item['expected']:>8.2f} {item['claimed']:>8.2f} {str(item['valid']):>6}")  # 输出当前 stage 的期望与声称值。
    print(f"  -> process={process_score:.1f}, outcome={outcome:.1f}")  # 汇总当前候选的双维奖励。

bill-01 Process Verifier 账本
candidate stage      expected  claimed  valid
hack      subtotal    360.00   370.00  False
hack      total       333.00   324.00  False
  -> process=0.0, outcome=1.0
good      subtotal    360.00   360.00   True
good      total       324.00   324.00   True
  -> process=1.0, outcome=1.0


## 4. 结果表与结果解读

Process-guided 在相同候选上选择过程分更高的 good。两种方法最终金额准确率都为 100%，只有过程有效率揭示 outcome-only 的虚假成功。

In [4]:
process_choices = []  # 保存五张账单的过程引导选择。
for case in cases:  # 对同一批候选按过程分和最终分排序。
    scored = [(candidate, *verify_candidate(case, candidate)[:2]) for candidate in candidates[case["id"]]]  # 计算每个候选的过程分和 outcome。
    chosen_candidate, chosen_process, chosen_outcome = max(scored, key=lambda item: (item[1], item[2], item[0]["id"]))  # 优先选择过程连续的候选。
    process_choices.append({"case": case["id"], "chosen": chosen_candidate["id"], "process": chosen_process, "outcome": chosen_outcome})  # 保存双维选择结果。
baseline_outcome_accuracy = sum(choice["outcome"] for choice in baseline_choices) / len(cases)  # 计算 outcome-only 最终准确率。
baseline_process_validity = sum(verify_candidate(case, candidates[case["id"]][0])[0] for case in cases) / len(cases)  # 计算被基线选中 hack 的过程有效率。
guided_outcome_accuracy = sum(choice["outcome"] for choice in process_choices) / len(cases)  # 计算过程引导最终准确率。
guided_process_validity = sum(choice["process"] for choice in process_choices) / len(cases)  # 计算过程引导选择的过程有效率。
print("策略              outcome准确率  process有效率  选择")  # 输出两种选择策略同指标对照表头。
print(f"{'Outcome-only':<17} {baseline_outcome_accuracy:>13.1%} {baseline_process_validity:>14.1%}  {[choice['chosen'] for choice in baseline_choices]}")  # 输出只看最终答案的虚假成功。
print(f"{'Process-guided':<17} {guided_outcome_accuracy:>13.1%} {guided_process_validity:>14.1%}  {[choice['chosen'] for choice in process_choices]}")  # 输出过程引导的选择。
print("结果解读：最终准确率无法区分两种策略；逐步状态连续性让五条 hack 轨迹失去选择优势。")  # 明确为何需要过程指标。

策略              outcome准确率  process有效率  选择
Outcome-only             100.0%           0.0%  ['hack', 'hack', 'hack', 'hack', 'hack']
Process-guided           100.0%         100.0%  ['good', 'good', 'good', 'good', 'good']
结果解读：最终准确率无法区分两种策略；逐步状态连续性让五条 hack 轨迹失去选择优势。


## 5. 失败案例与修正

如果过程奖励只是“每个看似正确步骤加一”，重复正确 subtotal 可以刷高总分。修正要求 stage 序列恰好为 subtotal、total，并拒绝重复或乱序。

In [5]:
good_candidate = candidates[first_case["id"]][1]  # 读取第一张账单的正常两步轨迹。
repeated_candidate = {"id": "repeat-hack", "steps": [good_candidate["steps"][0], good_candidate["steps"][0], good_candidate["steps"][0], good_candidate["steps"][1]], "final": good_candidate["final"]}  # 构造重复小计步骤刷分的轨迹。
def naive_step_sum(case, candidate):  # 模拟只累加局部正确步骤的脆弱过程奖励。
    expected_subtotal = round(case["quantity"] * case["unit_price"], 2)  # 计算小计真值供每步独立判断。
    expected_total = case["truth"]  # 读取最终金额真值。
    return sum(step["stage"] == "subtotal" and abs(step["claimed"] - expected_subtotal) < 1.0e-9 or step["stage"] == "total" and abs(step["claimed"] - expected_total) < 1.0e-9 for step in candidate["steps"])  # 累加所有看似正确步骤且不检查重复。
def strict_stage_reward(case, candidate):  # 对必要 stage、顺序和数量实施结构门禁。
    stages = [step["stage"] for step in candidate["steps"]]  # 读取候选完整 stage 序列。
    if stages != ["subtotal", "total"]:  # 拒绝缺失、重复或乱序步骤。
        return 0.0  # 对结构不合法轨迹返回零过程分。
    return verify_candidate(case, candidate)[0]  # 对合法两步轨迹复用状态连续 verifier。
print(f"错误行为：普通good步数奖励={naive_step_sum(first_case, good_candidate)}，重复轨迹奖励={naive_step_sum(first_case, repeated_candidate)}")  # 展示重复步骤获得更高错误奖励。
print(f"修正行为：严格stage good={strict_stage_reward(first_case, good_candidate):.1f}，重复轨迹={strict_stage_reward(first_case, repeated_candidate):.1f}")  # 展示序列门禁消除刷分。

错误行为：普通good步数奖励=2，重复轨迹奖励=4
修正行为：严格stage good=1.0，重复轨迹=0.0


## 6. 生产边界

规则 verifier 只适合结构化算术。开放式推理需要学习型 PRM、符号工具或多 verifier 交叉检查，并评估步骤等价、误拒、reward hacking、分布外输入和 verifier 推理成本。

In [6]:
verifier_summary = {"cases": len(cases), "candidate_paths": sum(len(options) for options in candidates.values()), "baseline_process_validity": baseline_process_validity, "guided_process_validity": guided_process_validity, "external_actions": 0}  # 汇总本实验的过程评测范围和零外部副作用。
print("Verifier 生产交接快照：", verifier_summary)  # 输出需要被真实 PRM 和沙箱执行替换的指标。

Verifier 生产交接快照： {'cases': 5, 'candidate_paths': 10, 'baseline_process_validity': 0.0, 'guided_process_validity': 1.0, 'external_actions': 0}


## 7. 最小回归测试

只验证案例规模、outcome 平票、过程选择和防重复门禁。

In [7]:
assert len(cases) >= 5  # 保证案例至少包含五张有业务字段的账单。
assert baseline_outcome_accuracy == guided_outcome_accuracy == 1.0  # 保证两种策略最终答案相同以隔离过程评测差异。
assert guided_process_validity > baseline_process_validity  # 保证过程 verifier 改善被选轨迹的有效性。
assert all(choice["chosen"] == "good" for choice in process_choices)  # 保证过程引导选择状态连续轨迹。
assert naive_step_sum(first_case, repeated_candidate) > naive_step_sum(first_case, good_candidate)  # 保证失败案例真实复现重复步骤刷分。
assert strict_stage_reward(first_case, repeated_candidate) == 0.0  # 保证严格 stage 门禁拒绝重复轨迹。